# 04 Evaluate

This notebook calls the APIM endpoint for each dataset row and then runs built-in Azure AI Evaluation metrics.

## Load Environment and Dataset

In [ ]:
import json
import os
from pathlib import Path

from azure.ai.evaluation import (
    CoherenceEvaluator,
    FluencyEvaluator,
    OpenAIModelConfiguration,
    SimilarityEvaluator,
    evaluate,
 )
from dotenv import load_dotenv

env_path = Path('../env/.env')
if not env_path.exists():
    raise RuntimeError('env/.env not found. Run 01_deploy_infra.ipynb first.')

load_dotenv(env_path)

EVAL_DATASET_PATH = os.getenv('EVAL_DATASET_PATH', '../data/eval-prompts.jsonl')
EVAL_OUTPUT_PATH = os.getenv('EVAL_OUTPUT_PATH', '../outputs/eval-results.json')
APIM_ENDPOINT = os.getenv('APIM_ENDPOINT', '').rstrip('/')
APIM_PATH = os.getenv('APIM_CHAT_COMPLETIONS_PATH', '/openai/chat/completions')
APIM_API_KEY = os.getenv('APIM_API_KEY', '')

AOAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', '').rstrip('/')
AOAI_KEY = os.getenv('AZURE_OPENAI_API_KEY', '')
AOAI_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'chat4o')

# Evaluator is routed through APIM using OpenAI-compatible base URL.
# Example: /openai/chat/completions -> base URL /openai
APIM_EVAL_BASE_URL = f"{APIM_ENDPOINT}{APIM_PATH.replace('/chat/completions', '')}"
APIM_EVAL_MODEL = os.getenv('APIM_EVAL_MODEL', AOAI_DEPLOYMENT)

dataset_path = Path(EVAL_DATASET_PATH)
if not dataset_path.exists():
    raise RuntimeError(f'Dataset file not found: {dataset_path}')

dataset = [json.loads(line) for line in dataset_path.read_text(encoding='utf-8').splitlines() if line.strip()]
print(f'Loaded dataset rows: {len(dataset)}')
print(f'Evaluator via APIM base URL: {APIM_EVAL_BASE_URL}')

## Generate Responses Through APIM

In [ ]:
import requests

if not APIM_API_KEY or APIM_API_KEY.startswith('[Retrieve from Azure Portal'):
    raise RuntimeError('APIM_API_KEY is not set with a real key in env/.env')

url = f"{APIM_ENDPOINT}{APIM_PATH}"
headers = {
    'Ocp-Apim-Subscription-Key': APIM_API_KEY,
    'Content-Type': 'application/json'
}

eval_rows = []
for row in dataset:
    prompt = row.get('query', '')
    payload = {
        'messages': [
            {'role': 'system', 'content': 'You are a concise assistant.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_completion_tokens': 140
    }

    resp = requests.post(url, headers=headers, json=payload, timeout=45)
    if resp.status_code == 200:
        data = resp.json()
        model_response = data.get('choices', [{}])[0].get('message', {}).get('content', '')
    else:
        model_response = f'ERROR {resp.status_code}: {resp.text[:300]}'

    eval_rows.append({
        'query': prompt,
        'response': model_response,
        'context': row.get('expected_behavior', '')
    })

# Persist the evaluator input dataset here so the next cell can directly consume it.
eval_input_path = Path('../outputs/eval-input.jsonl')
eval_input_path.parent.mkdir(parents=True, exist_ok=True)
with eval_input_path.open('w', encoding='utf-8') as f:
    for item in eval_rows:
        f.write(json.dumps(item) + '\n')

print(f'Collected APIM responses: {len(eval_rows)}')
print(f'Saved evaluator input: {eval_input_path}')

## Run Azure AI Evaluation

In [ ]:
# Route evaluator LLM calls through APIM (same gateway pattern as generation).
# OpenAIModelConfiguration in this SDK accepts: type, api_key, model, base_url, organization.
if not APIM_API_KEY or APIM_API_KEY.startswith('[Retrieve from Azure Portal'):
    raise RuntimeError('APIM_API_KEY is not set with a real key in env/.env')

eval_model_config = OpenAIModelConfiguration(
    type='openai',
    model=APIM_EVAL_MODEL,
    base_url=f"{APIM_EVAL_BASE_URL}?subscription-key={APIM_API_KEY}",
    api_key=APIM_API_KEY,  # required by schema; APIM policy can use query key
)

print('Evaluator model routing:')
print(f"  base_url: {APIM_EVAL_BASE_URL}")
print(f"  model:    {APIM_EVAL_MODEL}")
print('  auth:     APIM subscription key in query string')

coherence = CoherenceEvaluator(model_config=eval_model_config)
fluency = FluencyEvaluator(model_config=eval_model_config)
similarity = SimilarityEvaluator(model_config=eval_model_config)

if 'eval_input_path' not in globals():
    raise RuntimeError('eval_input_path is not defined. Run the APIM response generation cell first.')

result = evaluate(
    data=str(eval_input_path),
    evaluators={
        'coherence': coherence,
        'fluency': fluency,
        'similarity': similarity
    }
)

result

## Save Results

In [ ]:
output_file = Path(EVAL_OUTPUT_PATH)
output_file.parent.mkdir(parents=True, exist_ok=True)
output_file.write_text(json.dumps(result, indent=2), encoding='utf-8')
print(f'Saved evaluation output: {output_file}')